# Minimal TRL Training Example

This notebook demonstrates a minimal RL loop using HuggingFace TRL with a small model.

## 1. Install Dependencies

In [ ]:
!pip install transformers trl accelerate -q

## 2. Load Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token for GPT2
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print(f"Loaded {model_name}")

## 3. Create Dummy Environment

In [ ]:
class SimpleEnv:
    def reset(self):
        return "Emergency at location A"

    def step(self, action):
        if "ambulance" in action.lower():
            return "done", 10, True, {}
        return "done", -5, True, {}

env = SimpleEnv()
print("Environment initialized")

## 4. Run RL Loop

In [ ]:
for episode in range(5):
    obs = env.reset()
    
    prompt = f"Dispatch decision: {obs}"
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.7
    )
    
    action = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    _, reward, _, _ = env.step(action)
    
    print(f"Episode {episode}: Reward={reward}")
    print(f"  Prompt: {prompt}")
    print(f"  Action: {action[:50]}...")
    print()

## 5. Summary

This minimal example demonstrates:
- Loading a small causal LM (GPT-2)
- Creating a simple environment with rewards
- Running an RL loop that generates actions and receives rewards

For full TRL training, you would use `PPOTrainer` from the TRL library.